# Model 2: Scalar candidate features

Revised feature-engineering notebook for Step 1 outputs. Inputs: `train_sampled.parquet`, `val.parquet`, `test.parquet`, and `product_metadata_clean.parquet`.

This notebook produces the requested **numerical feature block** for Model 2; it does not reproduce the original DIN text/sequence embedding pipeline. No learned ID/term vocabulary is required. The L4 statistics' keys are learned from eligible training histories only. Metadata is an attribute lookup, not a source of fitted catalogue-wide statistics.

Outputs: `features/model2/channel_features/features_{train,val,test}.parquet`, one row per original slate, only `slate_id` plus candidate-aligned feature lists. No labels, raw candidate IDs, purchase flags or strata are saved. List index is zero-based `position`. The final helper reconstructs `(slate_id, position)` and performs a validated left join in the caller's original order. Never join by `slate_id` alone after exploding.

**Verified Step 1 contract:** `history_days_before` contains **days**, computed as the reverse cumulative sum of raw second-based gaps divided by 86,400. `HISTORY_AGE_UNIT="days"` converts these ages back to seconds internally. Set it to `"seconds"` only for an input age column already measured in seconds. Do not pass `history_time_list`: those are inter-event gaps, with one extra entry, not aligned ages before the search. The earlier conservative minimum-age buffer is retained: **strictly more than 864 seconds** (0.01 days) before the current search. Events exactly 864 seconds old are excluded. The shared filter applies consistently to ratios, other channel features and fitted L4 statistics. Its necessity depends on the logging process; pre-query events are not automatically leakage. Catalogue attributes must be available at prediction time; a static catalogue cannot prove point-in-time correctness.


In [ ]:
# In Colab, run this once if dependencies are unavailable:
# %pip install numpy pandas pyarrow
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass
from pathlib import Path
from collections import Counter, defaultdict
import json, math, os, uuid
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

PROJECT = Path('/content/drive/MyDrive/Grp8Submission')  # edit for local use
DATA_DIR = PROJECT / 'step1'                  # step 1 outputs
OUT_DIR = PROJECT / 'step2_model2_scalar'     # this notebook's outputs
INPUTS = {s: DATA_DIR / f'{f}.parquet' for s, f in
          [('train', 'train_sampled'), ('val', 'val'), ('test', 'test')]}
WEIGHT = {"CLICK": 1, "FLW": 2, "CART": 5, "ORD": 10}
METADATA_PATH = DATA_DIR / 'product_metadata_clean.parquet'
COL = {'slate': 'slate_id', 'hist': 'history_wid_list',
       'action': 'history_type_list', 'age': 'history_days_before',
       'candidate': 'candidate_wid_list', 'cand_flag': 'candidate_has_metadata'}
CHANNELS = {'click': ({'CLICK'}, 7*86400),
            'cart': ({'CART', 'FLW'}, 90*86400),
            'order': ({'ORD'}, 365*86400)}
HISTORY_AGE_UNIT = "days"  # verified from step1_data_preprocessing.ipynb
MIN_AGE_DAYS = 0.01
MIN_AGE_SECONDS = MIN_AGE_DAYS * 86400
BATCH_SIZE = 512  # slate rows; each row may contain many events
OVERWRITE = True
NAN = float('nan')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Feature definitions

All four behavioural features are computed **independently per channel**, without truncation or mixing channels. Every eligible repeated event counts.

| Feature suffix | Definition |
|---|---|
| `brand_affinity_ratio` | Matching-brand event count / eligible events with known brand. NaN if candidate brand is missing or denominator is zero. |
| `cate_affinity_ratio` | Matching-L4 event count / eligible events with known L4. Same missing rule. |
| `dominant_brand_match` | 1 if candidate brand equals the most frequent known history brand; 0 otherwise. Ties use smallest raw brand ID. NaN if either side is unavailable. |
| `hist_item_count` | Number of eligible events matching the candidate product. NaN if candidate ID is missing, history is unavailable, or a nonempty channel has no known historical item IDs; 0 for an observed empty channel or no match. |

Missing raw IDs (`-1` or null) never match each other. Unseen but known raw IDs still compare normally; they are not collapsed into a shared unknown brand. Partial histories use known observations, with flags identifying incompleteness.

### `missing_value_flags` implementation

These are separate numeric columns, not a packed bitmask. Each has prefix `missing_value_flags_`; 1 denotes the condition, 0 its absence.

- Candidate: `candidate_item`, `candidate_metadata`, `candidate_brand`, `candidate_cate`.
- History: `history_unavailable` (null list, distinct from observed empty list), `history_invalid_event` (at least one unknown action or nonfinite/missing age).
- Each channel: `empty`, `any_missing_item`, `any_missing_brand`, `any_missing_cate`, `no_known_item`, `no_known_brand`, `no_known_cate`.
- Each channel/L4 pair: `l4_stats_unavailable` (candidate category missing or unseen in that channel's eligible training histories).

A provided candidate metadata flag of 0, -1 or null forces candidate attributes to missing, even if the catalogue has a row. An absent flag column falls back to the catalogue. Flags are always defined; for an unavailable history the empty flag is 0, while no-known flags are 1.

### `per_L4_cate_descriptive_statistics` implementation

Fit one dictionary per channel from **train_sampled histories only**, after exactly the same time filters. For each known L4 category, count eligible occurrences and describe their ages. Attach the candidate's category statistics under `<channel>_l4_*`:

- `event_count`: number of observed history occurrences (repeated appearances across slates count again).
- `slate_count`: training slates with at least one such event.
- `events_per_slate_mean`, `events_per_slate_std`: mean and population standard deviation of category event counts across training slates with available histories, **including zeros**.
- `age_seconds_mean`, `age_seconds_std`, `age_seconds_min`, `age_seconds_max`: occurrence-weighted age summaries; standard deviation uses `ddof=0`.

Missing/unseen category statistics are NaN, never fabricated zeros. No purchase labels, target encodings, validation/test frequencies, global metadata frequencies, scalers or imputers are fitted. Training statistics are unsupervised full-training summaries (not out-of-fold). For strict historical online backtesting, fit on an earlier time cutoff instead. Overlapping slate histories make these exposure counts, not unique physical-event counts: no unique event ID is supplied.


In [61]:
def raw_id(value):
    if value is None or pd.isna(value):
        return None
    if isinstance(value, (bool, np.bool_)):
        raise ValueError('Boolean supplied as an ID')
    try:
        n = int(value)
    except (ValueError, TypeError, OverflowError) as e:
        raise ValueError(f'Invalid integer ID: {value!r}') from e
    if isinstance(value, (float, np.floating)) and n != value:
        raise ValueError(f'Non-integral ID: {value!r}')
    if n == -1:
        return None
    if n < 0:
        raise ValueError(f'Unexpected negative ID: {value!r}')
    return n

class ProductLookup:
    """Three-column indexed DataFrame; no SQLite and no whole-catalogue statistics."""
    def __init__(self, frame):
        frame = frame[['wid', 'brand_id', 'cate_id_4']].copy()
        for c in frame:
            # Exact nullable integer conversion; no float intermediate for large IDs.
            frame[c] = pd.array(frame[c], dtype='Int64')
            frame[c] = frame[c].mask(frame[c] == -1)
            if (frame[c].dropna() < 0).any():
                raise ValueError(f'Unexpected negative ID in {c}')
        if frame.wid.isna().any() or frame.wid.duplicated().any():
            raise ValueError('Metadata requires one nonmissing row per wid')
        self.frame = frame.set_index('wid')

    def resolve(self, ids):
        ids = sorted({x for x in ids if x is not None})
        if not ids:
            return {}
        found = self.frame.reindex(ids)
        exists = pd.Index(ids).isin(self.frame.index)
        return {w: (raw_id(b), raw_id(c), bool(hit))
                for w, b, c, hit in zip(ids, found.brand_id, found.cate_id_4, exists)}

def sequence(value, name, nullable=False):
    if value is None:
        if nullable:
            return None
        raise ValueError(f'{name} cannot be null')
    if not isinstance(value, (list, tuple, np.ndarray)):
        raise ValueError(f'{name} must be a list')
    return list(value)

def parse_row(row):
    if HISTORY_AGE_UNIT not in ("days", "seconds"):
        raise ValueError("HISTORY_AGE_UNIT must be days or seconds")
    age_scale = 86400.0 if HISTORY_AGE_UNIT == "days" else 1.0
    candidates = [raw_id(x) for x in sequence(row[COL['candidate']], COL['candidate'])]
    flags = row.get(COL['cand_flag'])
    if COL['cand_flag'] in row:
        flags = [None]*len(candidates) if flags is None else sequence(flags, COL['cand_flag'])
        if len(flags) != len(candidates):
            raise ValueError('Candidate metadata flags are misaligned')
        # Boolean metadata flags are valid; product IDs still reject booleans.
        flags = [int(x) if isinstance(x, (bool, np.bool_)) else raw_id(x)
                 for x in flags]
        if any(x not in (None, 0, 1) for x in flags):
            raise ValueError('Candidate metadata flags must be 0, 1, -1 or null')
    history = [sequence(row[COL[k]], COL[k], nullable=True) for k in ('hist', 'action', 'age')]
    unavailable = any(x is None for x in history)
    channels = {ch: [] for ch in CHANNELS}
    invalid = False
    if unavailable:
        if not all(x is None for x in history):
            raise ValueError('Partially null history columns: cannot establish alignment')
    else:
        if len({len(x) for x in history}) != 1:
            raise ValueError('History item/action/time lengths differ')
        for w, action, age in zip(*history):
            w = raw_id(w)
            try:
                seconds = float(age) * age_scale
            except (TypeError, ValueError):
                seconds = NAN
            ch = next((k for k, (actions, _) in CHANNELS.items() if action in actions), None)
            if ch is None or not math.isfinite(seconds):
                invalid = True
                continue
            if MIN_AGE_SECONDS < seconds <= CHANNELS[ch][1]:
                channels[ch].append((w, seconds, WEIGHT[action]))
    return candidates, flags, channels, unavailable, invalid

def iter_source(path):
    seen = set()
    with pq.ParquetFile(path) as pf:
        required = [COL[k] for k in ('slate', 'hist', 'action', 'age', 'candidate')]
        missing = set(required) - set(pf.schema_arrow.names)
        if missing:
            raise ValueError(f'{path}: missing {sorted(missing)}')
        selected = required + ([COL['cand_flag']] if COL['cand_flag'] in pf.schema_arrow.names else [])
        for batch in pf.iter_batches(batch_size=BATCH_SIZE, columns=selected):
            parsed = []
            for row in batch.to_pylist():
                sid = row[COL['slate']]
                if sid is None or sid in seen:
                    raise ValueError(f'{path}: missing or duplicate slate_id {sid!r}')
                seen.add(sid)
                parsed.append((sid, parse_row(row)))
            yield parsed

def batch_lookup(parsed, product):
    ids = set()
    for _, (cands, _, channels, _, _) in parsed:
        ids.update(cands)
        for events in channels.values():
            ids.update(w for w, _, _ in events)
    return product.resolve(ids)


In [62]:
STAT_NAMES = ['event_count', 'slate_count', 'events_per_slate_mean',
              'events_per_slate_std', 'age_seconds_mean', 'age_seconds_std',
              'age_seconds_min', 'age_seconds_max']

def fit_l4_statistics(train_path, product):
    # Store count moments and Welford age moments; memory scales with categories.
    acc = {ch: {} for ch in CHANNELS}
    n_available = 0
    for parsed in iter_source(train_path):
        lookup = batch_lookup(parsed, product)
        for _, (_, _, channels, unavailable, _) in parsed:
            if unavailable:
                continue
            n_available += 1
            for ch, events in channels.items():
                counts = Counter()
                for w, age, _ in events:
                    cate = lookup.get(w, (None, None, False))[1]
                    if cate is None:
                        continue
                    a = acc[ch].setdefault(cate, [0, 0, 0, 0.0, 0.0, math.inf, -math.inf])
                    a[0] += 1
                    delta = age - a[3]
                    a[3] += delta / a[0]
                    a[4] += delta * (age - a[3])
                    a[5], a[6] = min(a[5], age), max(a[6], age)
                    counts[cate] += 1
                for cate, count in counts.items():
                    acc[ch][cate][1] += 1
                    acc[ch][cate][2] += count*count
    result = {ch: {} for ch in CHANNELS}
    for ch, categories in acc.items():
        for cate, (n, slates, squares, mean, m2, lo, hi) in categories.items():
            count_mean = n / n_available
            result[ch][cate] = [float(n), float(slates), count_mean,
                math.sqrt(max(0.0, squares/n_available-count_mean**2)),
                mean, math.sqrt(max(0.0, m2/n)), lo, hi]
    return result, n_available

BASE_FLAGS = ['candidate_item', 'candidate_metadata', 'candidate_brand', 'candidate_cate',
              'history_unavailable', 'history_invalid_event']
CH_FLAGS = ['empty', 'any_missing_item', 'any_missing_brand', 'any_missing_cate',
            'no_known_item', 'no_known_brand', 'no_known_cate', 'l4_stats_unavailable']
BEHAVIOUR = ['brand_affinity_ratio', 'cate_affinity_ratio', 'dominant_brand_match', 'hist_item_count']
FEATURES = ['missing_value_flags_'+x for x in BASE_FLAGS]
for ch in CHANNELS:
    FEATURES += [f'{ch}_{x}' for x in BEHAVIOUR]
    FEATURES += [f'missing_value_flags_{ch}_{x}' for x in CH_FLAGS]
    FEATURES += [f'{ch}_l4_{x}' for x in STAT_NAMES]

def engineer_row(parsed, lookup, fitted):
    candidates, flags, channels, unavailable, invalid = parsed
    out = {name: [] for name in FEATURES}
    summaries = {}
    for ch, events in channels.items():
        items, brands, cates = Counter(), Counter(), Counter()
        brand_weights, cate_weights = Counter(), Counter()
        for w, _, weight in events:
            b, c, _ = lookup.get(w, (None, None, False))
            if w is not None:
                items[w] += 1
            if b is not None:
                brands[b] += 1
                brand_weights[b] += weight
            if c is not None:
                cates[c] += 1
                cate_weights[c] += weight
        dominant = (
            min(brand_weights, key=lambda b: (-brand_weights[b], b))
            if brand_weights else None
        )
        summaries[ch] = (
            len(events), items, brands, cates, dominant,
            brand_weights, cate_weights,
        )
    for i, w in enumerate(candidates):
        b, c, exists = lookup.get(w, (None, None, False))
        if flags is not None and flags[i] != 1:
            b, c, exists = None, None, False
        values = dict(zip(['missing_value_flags_'+x for x in BASE_FLAGS],
                          [w is None, not exists, b is None, c is None, unavailable, invalid]))
        for ch, (
            n, items, brands, cates, dominant,
            brand_weights, cate_weights,
        ) in summaries.items():
            nb = sum(brands.values())
            nc = sum(cates.values())
            ni = sum(items.values())
            brand_denominator = sum(brand_weights.values())
            cate_denominator = sum(cate_weights.values())

            values[f"{ch}_brand_affinity_ratio"] = (
                brand_weights[b] / brand_denominator
                if b is not None and brand_denominator > 0
                else NAN
            )

            values[f"{ch}_cate_affinity_ratio"] = (
                cate_weights[c] / cate_denominator
                if c is not None and cate_denominator > 0
                else NAN
            )
            values[f"{ch}_dominant_brand_match"] = (
                float(b == dominant)
                if b is not None and dominant is not None
                else NAN
            )
            values[f'{ch}_hist_item_count'] = float(items[w]) if w is not None and not unavailable and (n == 0 or ni > 0) else NAN
            stats = fitted[ch].get(c)
            flag_values = [n == 0 and not unavailable, ni < n, nb < n, nc < n,
                           ni == 0, nb == 0, nc == 0, stats is None]
            values.update(zip([f'missing_value_flags_{ch}_{x}' for x in CH_FLAGS], flag_values))
            values.update(zip([f'{ch}_l4_{x}' for x in STAT_NAMES], stats if stats is not None else [NAN]*len(STAT_NAMES)))
        for name in FEATURES:
            out[name].append(float(values[name]))
    return out


## Output order, joins and validation

The reader follows physical input row order; every output feature list follows the unchanged candidate list. Duplicate candidate products remain separate positions. Empty candidate slates remain in the saved file with empty lists and yield no candidate rows when exploded. Duplicate slate IDs, mismatched list lengths and malformed IDs fail loudly.

Only the three split Parquet files are published. Definitions are embedded in their schema metadata. The fitted L4 statistics remain in memory and are refitted from train on rerun. Existing outputs are protected unless `OVERWRITE=True`. Writes use temporary files and atomic replacement per split; this is not a three-file transaction.

Metadata uses an indexed DataFrame containing only three columns, with batch lookups. It must fit RAM; unlike SQLite, this design has no disk-backed lookup. Behaviour inputs and outputs are streamed. A set of slate IDs is retained per pass to check uniqueness.


In [63]:
def write_split(path, destination, product, fitted, n_train):
    destination = Path(destination)
    if destination.exists() and not OVERWRITE:
        raise FileExistsError(destination)
    with pq.ParquetFile(path) as pf:
        sid_type = pf.schema_arrow.field(COL['slate']).type
    metadata = {b'model2_feature_spec': json.dumps({
        'version': 3, 'input_age_unit': HISTORY_AGE_UNIT, 'min_age_inclusive': False, 'age_unit': 'seconds', 'min_age_seconds': MIN_AGE_SECONDS,
        'windows_seconds': {k: v[1] for k, v in CHANNELS.items()},
        'position': 'zero-based candidate list index; Step 1 order unchanged',
        'fit_split': 'train_sampled', 'train_available_history_slates': n_train,
        'ratios': 'matching known attribute events / all known attribute events in channel',
        'missing_raw_id': -1, 'undefined_features': 'NaN', 'zero': 'computed no-match/count zero'
    }).encode()}
    # Parquet's compliant LIST encoding names its child 'element'.
    # Specify it explicitly so metadata-aware schema round trips are stable.
    schema = pa.schema([pa.field('slate_id', sid_type)] +
                       [pa.field(f, pa.list_(pa.field("element", pa.float64()))) for f in FEATURES], metadata=metadata)
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(destination.name+'.tmp-'+uuid.uuid4().hex)
    count = candidate_count = 0
    try:
        with pq.ParquetWriter(temporary, schema, compression='zstd') as writer:
            for parsed in iter_source(path):
                lookup = batch_lookup(parsed, product)
                records = []
                for sid, data in parsed:
                    features = engineer_row(data, lookup, fitted)
                    assert all(len(v) == len(data[0]) for v in features.values())
                    records.append({'slate_id': sid, **features})
                    candidate_count += len(data[0])
                writer.write_table(pa.Table.from_pylist(records, schema=schema))
                count += len(records)
        with pq.ParquetFile(temporary) as pf:
            assert pf.metadata.num_rows == count
            assert pf.schema_arrow.equals(schema, check_metadata=True), (
                f"Parquet schema mismatch. Expected: {schema}; read back: {pf.schema_arrow}"
            )
        if destination.exists() and not OVERWRITE:
            raise FileExistsError(destination)
        os.replace(temporary, destination)
    finally:
        temporary.unlink(missing_ok=True)
    print(f'{destination.name}: {count:,} slates, {candidate_count:,} candidates')

def explode_features(table):
    """Reconstruct explicit join keys; intended for integration on manageable tables."""
    rows = []
    for row in table.to_pylist():
        lengths = {len(row[f]) for f in FEATURES}
        if len(lengths) != 1:
            raise ValueError('Misaligned feature lists')
        for position in range(next(iter(lengths))):
            rows.append({'slate_id': row['slate_id'], 'position': position,
                         **{f: row[f][position] for f in FEATURES}})
    frame = pd.DataFrame(rows, columns=['slate_id', 'position']+FEATURES)
    if frame.duplicated(['slate_id', 'position']).any():
        raise ValueError('Duplicate join keys')
    return frame

def join_in_step1_order(step1_candidates, feature_table):
    """step1_candidates must already contain slate_id and zero-based position."""
    keys = ['slate_id', 'position']
    if step1_candidates.duplicated(keys).any():
        raise ValueError('Duplicate Step 1 candidate keys')
    if set(FEATURES) & set(step1_candidates.columns):
        raise ValueError('Feature columns already present')
    right = explode_features(feature_table)
    # Check the exact key universe before joining, including extra feature rows.
    check = step1_candidates[keys].merge(right[keys], on=keys, how='outer',
                                        validate='one_to_one', indicator=True)
    if not check['_merge'].eq('both').all():
        raise ValueError('Step 1 and feature key sets differ')
    marker = '__model2_original_order__'
    if marker in step1_candidates:
        raise ValueError('Reserved temporary column collision')
    left = step1_candidates.copy()
    left[marker] = np.arange(len(left))
    joined = left.merge(right, on=keys, how='left', sort=False, validate='one_to_one')
    joined = joined.sort_values(marker, kind='stable').drop(columns=marker).reset_index(drop=True)
    assert joined[keys].equals(step1_candidates[keys].reset_index(drop=True))
    return joined


## Synthetic checks (no real input files needed)

These test inclusive time boundaries, the minimum-age buffer, FLW grouping, duplicate events/candidates, missing versus empty histories, dominant-brand tie breaking, statistics with zero-count slates, train-only fitting, and a Parquet round trip plus order-preserving join.


In [64]:
def self_test():
    from tempfile import TemporaryDirectory
    product = ProductLookup(pd.DataFrame({'wid': [1, 2, 3, 4],
        'brand_id': [10, 20, -1, 40], 'cate_id_4': [100, 200, -1, 400]}))
    def row(sid, items, actions, ages, cands=(1, 2, 3, 1)):
        return {'slate_id': sid, COL['hist']: items, COL['action']: actions,
                COL['age']: (None if ages is None else [
                    None if x is None else x / (86400.0 if HISTORY_AGE_UNIT == 'days' else 1.0)
                    for x in ages]), COL['candidate']: list(cands)}
    a = row(9, [1,2,1,2,1,1,1,1,2,2,1,1],
        ['CLICK']*5+['FLW','CART','ORD','CART','ORD','CLICK','CLICK'],
        [865,7*86400,863,7*86400+1,-10,90*86400,90*86400+1,
         365*86400,865,365*86400+1,None,float('inf')])
    b = row(3, [], [], [], (4,))
    with TemporaryDirectory() as d:
        train, output = Path(d)/'train.parquet', Path(d)/'out.parquet'
        pq.write_table(pa.Table.from_pylist([a,b]), train)
        fitted, n = fit_l4_statistics(train, product)
        assert n == 2
        assert fitted['click'][100][:4] == [1.,1.,.5,.5]
        assert 400 not in fitted['click']  # train candidate cannot create history stats
        boundary = parse_row(row(100,[1,1,1],['CLICK']*3,[863.999,864.0,864.001],(1,)))
        assert len(boundary[2]['click']) == 1
        assert np.isclose(boundary[2]['click'][0][1],864.001)
        if HISTORY_AGE_UNIT == 'days':
            day_row = row(101,[1,1,1,1],['CLICK']*4,[0]*4,(1,))
            day_row[COL['age']] = [0.005,0.01,1.0,7.0]
            assert parse_row(day_row)[2]["click"] == [
                (1, 86400.0, 1),
                (1, 604800.0, 1),
            ]
        parsed = parse_row(a)
        lookup = product.resolve([1,2,3,4])
        weighted_row = row(
            102,
            [1, 1, 2],
            ["FLW", "CART", "CART"],
            [865, 865, 865],  # row() converts seconds to the configured input unit
            (1, 2),
        )

        weighted_features = engineer_row(
            parse_row(weighted_row), lookup, fitted
        )

        assert np.allclose(
            weighted_features["cart_brand_affinity_ratio"],
            [7 / 12, 5 / 12],
        )

        assert np.allclose(
            weighted_features["cart_cate_affinity_ratio"],
            [7 / 12, 5 / 12],
        )
        f = engineer_row(parsed, lookup, fitted)
        assert f['click_brand_affinity_ratio'] == [.5,.5,NAN,.5] or np.allclose(f['click_brand_affinity_ratio'], [.5,.5,NAN,.5], equal_nan=True)
        assert f['click_dominant_brand_match'][:2] == [1.,0.]
        assert f['cart_hist_item_count'][:2] == [1.,1.]
        assert f['order_hist_item_count'][:2] == [1.,0.]
        assert f['missing_value_flags_history_invalid_event'] == [1.]*4
        empty = engineer_row(parse_row(b), lookup, fitted)
        assert empty['click_hist_item_count'] == [0.]
        assert math.isnan(empty['click_brand_affinity_ratio'][0])
        nullrow = row(5,None,None,None,(1,))
        assert math.isnan(engineer_row(parse_row(nullrow),lookup,fitted)['click_hist_item_count'][0])
        missing = row(6,[-1],['CLICK'],[865],(-1,1))
        mf = engineer_row(parse_row(missing),lookup,fitted)
        assert all(math.isnan(x) for x in mf['click_hist_item_count'])
        boolean_flags = dict(a, candidate_has_metadata=[False,True,np.bool_(False),np.bool_(True)])
        assert parse_row(boolean_flags)[1] == [0,1,0,1]
        mixed_flags = dict(a, candidate_has_metadata=[True,0,-1,None])
        assert parse_row(mixed_flags)[1] == [1,0,None,None]
        for bad_id in (True, np.bool_(False)):
            try:
                raw_id(bad_id)
            except ValueError:
                pass
            else:
                raise AssertionError('Boolean product ID was accepted')
        try:
            parse_row(dict(a, candidate_has_metadata=[2,1,1,1]))
        except ValueError:
            pass
        else:
            raise AssertionError('Invalid metadata flag was accepted')
        masked = dict(a, candidate_has_metadata=[0,1,1,1])
        assert math.isnan(engineer_row(parse_row(masked),lookup,fitted)['click_brand_affinity_ratio'][0])
        # Known candidate, known history, no match => 0, not NaN.
        assert engineer_row(parse_row(row(8,[1],['CLICK'],[865],(2,))),lookup,fitted)['click_brand_affinity_ratio'] == [0.]
        # Repeated events count; repeated candidates preserve positions.
        rf = engineer_row(parse_row(row(7,[1,1],['CLICK']*2,[865]*2,(1,1))),lookup,fitted)
        assert rf['click_hist_item_count'] == [2.,2.]
        before = repr(fitted)
        engineer_row(parse_row(row(99,[4],['CLICK'],[865],(4,))),lookup,fitted)
        assert repr(fitted) == before
        write_split(train,output,product,fitted,n)
        saved = pq.read_table(output)
        assert saved['slate_id'].to_pylist() == [9,3]
        assert saved.column_names == ['slate_id']+FEATURES
        flat = explode_features(saved)
        assert list(zip(flat.slate_id,flat.position)) == [(9,0),(9,1),(9,2),(9,3),(3,0)]
        base = flat[['slate_id','position']].iloc[::-1].reset_index(drop=True)
        joined = join_in_step1_order(base,saved)
        assert joined[['slate_id','position']].equals(base)
    print('All synthetic checks passed.')

self_test()


out.parquet: 2 slates, 5 candidates
All synthetic checks passed.


## Run on your data

Set the paths above, then run this cell. No labels are read. The tests above do not substitute for checking your actual input schema, age units, split chronology, or catalogue provenance. Training-only fitting prevents evaluation-set contamination; it does not establish that every training history was constructed without outcome leakage upstream.


In [65]:
def run_pipeline():
    for path in [*INPUTS.values(), METADATA_PATH]:
        if not path.is_file():
            raise FileNotFoundError(path)
    destinations = {s: OUT_DIR / f'features_{s}.parquet' for s in INPUTS}
    protected = {p.resolve() for p in [*INPUTS.values(), METADATA_PATH]}
    for dest in destinations.values():
        if dest.resolve() in protected:
            raise ValueError('Output would overwrite an input')
        if dest.exists() and not OVERWRITE:
            raise FileExistsError(f'{dest}; set OVERWRITE=True to replace your previous output')
    print('Loading three metadata columns...')
    # Ignore saved pandas dtypes (including unrelated nested-list columns).
    # Preserve nullable integer IDs without converting through floating point.
    with pq.ParquetFile(METADATA_PATH) as pf:
        metadata_table = pf.read(columns=['wid', 'brand_id', 'cate_id_4'])
    metadata_frame = metadata_table.to_pandas(
        ignore_metadata=True,
        types_mapper=lambda dtype: pd.Int64Dtype() if pa.types.is_integer(dtype) else None,
    )
    product = ProductLookup(metadata_frame)
    del metadata_table, metadata_frame
    print('Fitting L4 summaries from eligible training histories only...')
    fitted, n_train = fit_l4_statistics(INPUTS['train'], product)
    print('Known L4 categories per channel:', {ch:len(x) for ch,x in fitted.items()})
    for split, source in INPUTS.items():
        write_split(source, destinations[split], product, fitted, n_train)
    return destinations

outputs = run_pipeline()


Loading three metadata columns...
Fitting L4 summaries from eligible training histories only...
Known L4 categories per channel: {'click': 5879, 'cart': 6184, 'order': 5788}
features_train.parquet: 139,882 slates, 2,988,641 candidates
features_val.parquet: 16,924 slates, 1,511,179 candidates
features_test.parquet: 17,025 slates, 1,500,706 candidates


## Check Missingness

In [66]:
def summarise_missingness(paths, batch_size=1024):
    overview, records = [], []
    for split, path in paths.items():
        with pq.ParquetFile(path) as pf:
            flags = [c for c in pf.schema_arrow.names if c.startswith('missing_value_flags_')]
            if not flags:
                raise ValueError(f'{path}: no missingness flags found')
            counts = {f: [0, 0, 0] for f in flags}  # positives, slate-any, slate-all
            slates = nonempty = candidates = 0
            for batch in pf.iter_batches(columns=flags, batch_size=batch_size):
                for row in batch.to_pylist():
                    arrays = {f: np.asarray(row[f], dtype=float) for f in flags}
                    lengths = {len(a) for a in arrays.values()}
                    if len(lengths) != 1:
                        raise ValueError(f'{split}: misaligned flag lists')
                    size = next(iter(lengths))
                    slates += 1
                    candidates += size
                    nonempty += int(size > 0)
                    for f, a in arrays.items():
                        if not np.isin(a, [0., 1.]).all():
                            raise ValueError(f'{split}: {f} contains values other than 0/1')
                        counts[f][0] += int(a.sum())
                        counts[f][1] += int(size > 0 and a.any())
                        counts[f][2] += int(size > 0 and a.all())
            overview.append(dict(split=split, slates=slates, candidates=candidates,
                                 nonempty_slates=nonempty, empty_candidate_slates=slates-nonempty))
            for f, (positive, any_count, all_count) in counts.items():
                records.append(dict(split=split, flag=f, flagged_candidates=positive,
                    candidate_count=candidates, candidate_rate_pct=100*positive/candidates if candidates else np.nan,
                    flagged_slates_any=any_count, flagged_slates_all=all_count,
                    nonempty_slates=nonempty,
                    slate_any_rate_pct=100*any_count/nonempty if nonempty else np.nan,
                    slate_all_rate_pct=100*all_count/nonempty if nonempty else np.nan))
    detail = pd.DataFrame(records)
    candidate_rates = detail.pivot(index='flag', columns='split', values='candidate_rate_pct')
    slate_rates = detail.pivot(index='flag', columns='split', values='slate_any_rate_pct')
    candidate_rates = candidate_rates.reindex(columns=list(paths))
    slate_rates = slate_rates.reindex(columns=list(paths))
    drift = candidate_rates.copy()
    if 'train' in drift:
        for split in paths:
            if split != 'train':
                drift[f'{split}_minus_train_pp'] = drift[split]-drift['train']
    return pd.DataFrame(overview), detail, candidate_rates, slate_rates, drift

analysis_paths = {s: OUT_DIR / f'features_{s}.parquet' for s in INPUTS}
missingness_overview, missingness_detail, candidate_flag_rates, slate_flag_rates, flag_rate_comparison = summarise_missingness(analysis_paths)
try:
    from IPython.display import display
except ImportError:
    display = print
print('Coverage and denominators')
display(missingness_overview)
print('Candidate flag rates (%) and differences from train (percentage points)')
display(flag_rate_comparison.round(2))
print('Slates with at least one flagged candidate (%; excludes empty candidate slates)')
display(slate_flag_rates.round(2))
print('Full counts and rates are available in missingness_detail.')


Coverage and denominators


,split,slates,candidates,nonempty_slates,empty_candidate_slates
0,train,139882,2988641,139882,0
1,val,16924,1511179,16924,0
2,test,17025,1500706,17025,0


Candidate flag rates (%) and differences from train (percentage points)


split,train,val,test,val_minus_train_pp,test_minus_train_pp
flag,,,,,
missing_value_flags_candidate_brand,0.45,0.45,0.46,-0.01,0.00
missing_value_flags_candidate_cate,0.45,0.45,0.46,-0.01,0.00
missing_value_flags_candidate_item,0.00,0.00,0.00,0.00,0.00
missing_value_flags_candidate_metadata,0.45,0.45,0.46,-0.01,0.00
missing_value_flags_cart_any_missing_brand,59.12,59.68,59.37,0.56,0.25
missing_value_flags_cart_any_missing_cate,59.12,59.68,59.37,0.56,0.25
missing_value_flags_cart_any_missing_item,0.00,0.00,0.00,0.00,0.00
missing_value_flags_cart_empty,5.08,4.69,4.84,-0.39,-0.23
missing_value_flags_cart_l4_stats_unavailable,0.51,0.51,0.50,-0.00,-0.01


Slates with at least one flagged candidate (%; excludes empty candidate slates)


split,train,val,test
flag,,,
missing_value_flags_candidate_brand,6.87,18.57,18.55
missing_value_flags_candidate_cate,6.87,18.57,18.55
missing_value_flags_candidate_item,0.00,0.00,0.00
missing_value_flags_candidate_metadata,6.87,18.57,18.55
missing_value_flags_cart_any_missing_brand,59.07,59.77,59.82
missing_value_flags_cart_any_missing_cate,59.07,59.77,59.82
missing_value_flags_cart_any_missing_item,0.00,0.00,0.00
missing_value_flags_cart_empty,5.08,4.60,4.76
missing_value_flags_cart_l4_stats_unavailable,7.63,20.26,20.20


Full counts and rates are available in missingness_detail.


In [72]:
import pyarrow.parquet as pq

split = "train"  # "train", "val", or "test"
row_index = 0    # zero-based slate row in the feature file

feature_path = OUT_DIR / f"features_{split}.parquet"
input_path = INPUTS[split]  # train uses train_sampled.parquet

# Read the selected feature row.
with pq.ParquetFile(feature_path) as pf:
    if not 0 <= row_index < pf.metadata.num_rows:
        raise IndexError(
            f"row_index must be between 0 and {pf.metadata.num_rows - 1}"
        )

    offset = 0
    for batch in pf.iter_batches(batch_size=256):
        if row_index < offset + batch.num_rows:
            feature_row = batch.slice(row_index - offset, 1).to_pylist()[0]
            break
        offset += batch.num_rows

# Locate its original input row by slate_id.
slate_id = feature_row["slate_id"]
source_row = None

with pq.ParquetFile(input_path) as pf:
    for batch in pf.iter_batches(batch_size=256):
        slate_ids = batch.column(
            batch.schema.get_field_index("slate_id")
        ).to_pylist()

        if slate_id in slate_ids:
            position = slate_ids.index(slate_id)
            source_row = batch.slice(position, 1).to_pylist()[0]
            break

if source_row is None:
    raise ValueError(f"slate_id={slate_id} not found in {input_path}")

print("=== ORIGINAL STEP 1 DATA ===")
for name, value in source_row.items():
    print(f"{name}: {value}")

print("\n=== ENGINEERED FEATURES ===")
for name, value in feature_row.items():
    print(f"{name}: {value}")

=== ORIGINAL STEP 1 DATA ===
slate_id: 0
query: ['63995226', '60257419', '61002446', '25575117']
candidate_wid_list: [48645816, 60156606, 693764, 35010767, 42508190, 71429327, 52171878, 42608577, 82286960, 13527172, 29776696, 38870153, 50774882, 43323517, 36098226, 79988992, 98907906, 88380946, 52461499, 82122191, 84585257]
candidate_label_list: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0]
history_qry_list: ['-1', '-1', '-1', '68784030', '36547097\x1860257419\x183663289', '95922913\x1898423633', '95922913\x1898423633', '95922913\x1898423633', '94018157\x1895922913\x1898423633', '92191504\x1830479064\x1880855185', '97447992\x1899551658', '34535856\x1897447992', '97447992', '97447992', '97447992', '97447992', '97447992', '-1', '97447992', '97447992', '-1', '95922913\x1898423633', '95922913\x1898423633', '95922913\x1898423633', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '-1', '54279620\x1895922913', '54279620\x1895922913', '-1', '54279620\x1895922